# ARIMAX Hyperparameter Tuning on `micro_mobility_training_data_2025.csv`

Grid search ARIMAX (`SARIMAX` with exogenous vars) on a representative station subset, then final evaluate with best config.


## Colab Setup

- Auto-detect Colab and mount Google Drive.
- Update `COLAB_PROJECT_ROOT` if your Drive path is different.
- Keep station count small during tuning to control runtime.


## Step 1: Environment and Libraries

Loads libraries for ARIMAX modeling, grid search, and evaluation.


In [ ]:
from pathlib import Path
import json
import time
import warnings
import itertools
import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.statespace.sarimax import SARIMAX


## Step 2: Paths, Tuning Budget, and Grid

Sets data/artifact paths and ARIMAX search grid size.


In [ ]:
# Paths and config
COLAB_PROJECT_ROOT = Path('/content/drive/MyDrive/AIT/ML/citibike-netdemand-prediction')

IN_COLAB = False
try:
    import google.colab  # type: ignore
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = COLAB_PROJECT_ROOT
else:
    PROJECT_ROOT = Path.cwd().resolve().parents[1]

DATA_PATH = PROJECT_ROOT / 'data' / 'proceed' / 'micro_mobility_training_data_2025.csv'
ARTIFACT_DIR = PROJECT_ROOT / 'artifacts' / 'model_training' / 'arimax' / 'tuned_v1'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

HOLDOUT_DAYS = 7
VAL_DAYS = 14
TUNE_STATIONS = 12
FINAL_STATIONS = 30
MIN_TRAIN_POINTS = 24 * 30

# Compact grid (expand if needed)
P_LIST = [0, 1]
D_LIST = [0, 1]
Q_LIST = [0, 1]
SP_LIST = [0, 1]
SD_LIST = [0, 1]
SQ_LIST = [0, 1]
S_PERIOD = 24

print('IN_COLAB:', IN_COLAB)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATA_PATH:', DATA_PATH)
print('ARTIFACT_DIR:', ARTIFACT_DIR)

NOTEBOOK_T0 = time.perf_counter()


## Step 3: Data Load and Time Split

Builds chronological validation and test windows.


In [ ]:
data_prep_t0 = time.perf_counter()
# Load data
cols = [
    'station', 'date', 'hour', 'net_demand',
    'hour_sin', 'hour_cos', 'day_sin', 'day_cos',
    'lag_1h', 'lag_2h', 'lag_3h', 'lag_24h', 'rolling_mean_3h'
]

df = pd.read_csv(DATA_PATH, usecols=cols)
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.dropna(subset=['date', 'station', 'net_demand']).copy()
df['datetime_hour'] = df['date'] + pd.to_timedelta(df['hour'].astype(int), unit='h')

max_date = df['date'].max().normalize()
test_cutoff = max_date - pd.Timedelta(days=HOLDOUT_DAYS)
val_cutoff = test_cutoff - pd.Timedelta(days=VAL_DAYS)

print('Rows total:', len(df))
print('Val cutoff:', val_cutoff.date())
print('Test cutoff:', test_cutoff.date())

data_prep_seconds = float(time.perf_counter() - data_prep_t0)
print(f'Data prep time: {data_prep_seconds:.2f} sec ({data_prep_seconds/60:.2f} min)')


## Step 4: Station Selection

Chooses representative stations by activity magnitude.


In [ ]:
# Choose representative stations by activity magnitude
activity = df.groupby('station')['net_demand'].apply(lambda s: float(np.abs(s).sum())).sort_values(ascending=False)

tune_stations = activity.head(TUNE_STATIONS).index.tolist()
final_stations = activity.head(FINAL_STATIONS).index.tolist()

print('Tune stations:', len(tune_stations))
print('Final stations:', len(final_stations))
print(activity.head(10))


## Step 5: Utilities

Defines helper logic to evaluate a single config on one station.


In [ ]:
# Utilities
exog_cols = ['hour_sin', 'hour_cos', 'day_sin', 'day_cos', 'lag_1h', 'lag_2h', 'lag_3h', 'lag_24h', 'rolling_mean_3h']

warnings.filterwarnings('ignore')

def eval_config_on_station(sdf, order, seasonal_order):
    tr_mask = sdf['date'] <= val_cutoff
    va_mask = (sdf['date'] > val_cutoff) & (sdf['date'] <= test_cutoff)

    y_tr = sdf.loc[tr_mask, 'net_demand'].astype('float64')
    y_va = sdf.loc[va_mask, 'net_demand'].astype('float64')
    x_tr = sdf.loc[tr_mask, exog_cols].astype('float64')
    x_va = sdf.loc[va_mask, exog_cols].astype('float64')

    if len(y_tr) < MIN_TRAIN_POINTS or len(y_va) == 0:
        return None

    m = SARIMAX(
        endog=y_tr,
        exog=x_tr,
        order=order,
        seasonal_order=seasonal_order,
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    r = m.fit(disp=False)
    pred = r.forecast(steps=len(y_va), exog=x_va)
    mae = float(mean_absolute_error(y_va, pred))
    rmse = float(np.sqrt(mean_squared_error(y_va, pred)))
    return mae, rmse


## Step 6: Grid Search Tuning

Evaluates ARIMAX configs across tuning stations and records runtime.


In [ ]:
# Grid search on tuning stations
configs = []
for p,d,q in itertools.product(P_LIST, D_LIST, Q_LIST):
    for sp,sd,sq in itertools.product(SP_LIST, SD_LIST, SQ_LIST):
        configs.append(((p,d,q), (sp,sd,sq,S_PERIOD)))

print('Config count:', len(configs))

trial_rows = []
best = None
tuning_t0 = time.perf_counter()

for i, (order, seas) in enumerate(configs, start=1):
    t0 = time.perf_counter()
    maes = []
    rmses = []
    ok = 0

    for st in tune_stations:
        sdf = df.loc[df['station'] == st].sort_values('datetime_hour').copy()
        try:
            out = eval_config_on_station(sdf, order, seas)
            if out is None:
                continue
            mae, rmse = out
            maes.append(mae)
            rmses.append(rmse)
            ok += 1
        except Exception:
            continue

    if ok == 0:
        mean_mae = np.inf
        mean_rmse = np.inf
    else:
        mean_mae = float(np.mean(maes))
        mean_rmse = float(np.mean(rmses))

    elapsed = float(time.perf_counter() - t0)
    row = {
        'trial': i,
        'order': str(order),
        'seasonal_order': str(seas),
        'stations_ok': ok,
        'val_mae_mean': mean_mae,
        'val_rmse_mean': mean_rmse,
        'seconds': elapsed,
    }
    trial_rows.append(row)

    if ok > 0 and (best is None or mean_mae < best['val_mae_mean']):
        best = {'order': order, 'seasonal_order': seas, 'val_mae_mean': mean_mae, 'val_rmse_mean': mean_rmse}

    print(f"[{i}/{len(configs)}] order={order} seas={seas} ok={ok} val_mae={mean_mae:.4f} val_rmse={mean_rmse:.4f} time={elapsed:.1f}s")

trials_df = pd.DataFrame(trial_rows).sort_values('val_mae_mean')
print('Best configs:')
print(trials_df.head(5))

tuning_seconds = float(time.perf_counter() - tuning_t0)
print(f'Tuning grid time: {tuning_seconds:.2f} sec ({tuning_seconds/60:.2f} min)')


## Step 7: Final Evaluation with Best Config

Runs best ARIMAX config on final station set and reports test metrics/time.


In [ ]:
# Final evaluation on test window using best config
if best is None:
    raise RuntimeError('No valid ARIMAX config found. Reduce grid or MIN_TRAIN_POINTS.')

order = best['order']
seasonal_order = best['seasonal_order']

pred_parts = []
station_rows = []

loop_t0 = time.perf_counter()
for i, st in enumerate(final_stations, start=1):
    sdf = df.loc[df['station'] == st].sort_values('datetime_hour').copy()

    tr_mask = sdf['date'] <= test_cutoff
    te_mask = sdf['date'] > test_cutoff

    y_tr = sdf.loc[tr_mask, 'net_demand'].astype('float64')
    y_te = sdf.loc[te_mask, 'net_demand'].astype('float64')
    x_tr = sdf.loc[tr_mask, exog_cols].astype('float64')
    x_te = sdf.loc[te_mask, exog_cols].astype('float64')

    if len(y_tr) < MIN_TRAIN_POINTS or len(y_te) == 0:
        continue

    t0 = time.perf_counter()
    try:
        m = SARIMAX(
            endog=y_tr,
            exog=x_tr,
            order=order,
            seasonal_order=seasonal_order,
            enforce_stationarity=False,
            enforce_invertibility=False,
        )
        r = m.fit(disp=False)
        pred = r.forecast(steps=len(y_te), exog=x_te)

        st_mae = float(mean_absolute_error(y_te, pred))
        st_rmse = float(np.sqrt(mean_squared_error(y_te, pred)))
        st_sec = float(time.perf_counter() - t0)

        station_rows.append({'station': st, 'test_rows': int(len(y_te)), 'mae': st_mae, 'rmse': st_rmse, 'seconds': st_sec})
        pred_parts.append(pd.DataFrame({'station': st, 'date': sdf.loc[te_mask, 'date'].values, 'hour': sdf.loc[te_mask, 'hour'].astype(int).values, 'y_true': y_te.values, 'y_pred': np.asarray(pred)}))

        print(f"[{i}/{len(final_stations)}] OK {st} | MAE={st_mae:.4f} RMSE={st_rmse:.4f} time={st_sec:.1f}s")
    except Exception as e:
        print(f"[{i}/{len(final_stations)}] FAIL {st} | {e}")

loop_sec = float(time.perf_counter() - loop_t0)

if not pred_parts:
    raise RuntimeError('No station succeeded in final evaluation.')

pred_all = pd.concat(pred_parts, ignore_index=True)
station_df = pd.DataFrame(station_rows).sort_values('mae')

test_mae = float(mean_absolute_error(pred_all['y_true'], pred_all['y_pred']))
test_rmse = float(np.sqrt(mean_squared_error(pred_all['y_true'], pred_all['y_pred'])))

print(f'Final Global Test MAE : {test_mae:.4f}')
print(f'Final Global Test RMSE: {test_rmse:.4f}')
print(f'Final loop time: {loop_sec:.2f}s ({loop_sec/60:.2f} min)')

final_stage_seconds = float(loop_sec)
print(f'Final evaluation stage time: {final_stage_seconds:.2f} sec ({final_stage_seconds/60:.2f} min)')


## Step 8: Save Artifacts

Stores trials, predictions, station metrics, and timing summary.


In [ ]:
# Save artifacts
trials_path = ARTIFACT_DIR / 'trials.csv'
trials_df.to_csv(trials_path, index=False)

pred_path = ARTIFACT_DIR / 'predictions.csv'
pred_all.to_csv(pred_path, index=False)

station_path = ARTIFACT_DIR / 'station_metrics.csv'
station_df.to_csv(station_path, index=False)

total_notebook_seconds = float(time.perf_counter() - NOTEBOOK_T0)

metrics = {
    'data_path': str(DATA_PATH),
    'model': 'SARIMAX_ARIMAX_tuned',
    'holdout_days': HOLDOUT_DAYS,
    'val_days': VAL_DAYS,
    'tune_stations': TUNE_STATIONS,
    'final_stations': FINAL_STATIONS,
    'best_order': list(best['order']),
    'best_seasonal_order': list(best['seasonal_order']),
    'best_val_mae_mean': float(best['val_mae_mean']),
    'best_val_rmse_mean': float(best['val_rmse_mean']),
    'test_mae': test_mae,
    'test_rmse': test_rmse,
    'final_loop_seconds': loop_sec,
    'timing': {
        'data_prep_seconds': float(data_prep_seconds),
        'tuning_seconds': float(tuning_seconds),
        'final_stage_seconds': float(final_stage_seconds),
        'total_notebook_seconds': float(total_notebook_seconds),
    },
}
metrics_path = ARTIFACT_DIR / 'metrics.json'
with open(metrics_path, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

print('Saved:')
print('-', trials_path)
print('-', pred_path)
print('-', station_path)
print('-', metrics_path)

print(f"Total notebook runtime: {total_notebook_seconds:.2f} sec ({total_notebook_seconds/60:.2f} min)")
